# A5 — Nominal or Ordinal?

---

## 🔍 Problem-এ কী চাওয়া হয়েছে?

তিনটি feature দেওয়া আছে — প্রতিটির জন্য বলতে হবে **Nominal** নাকি **Ordinal**:

| Feature | Values |
|---|---|
| **(a)** `T_shirt_size` | {S, M, L, XL} |
| **(b)** `City` | {Dhaka, Chattogram, Rajshahi} |
| **(c)** `Satisfaction` | {Low, Medium, High} |


---

## 🎯 এই কাজ থেকে আমরা কী অর্জন করতে পারব?

- **Nominal** ও **Ordinal** data চেনার instinct তৈরি হবে।
- কোন feature-এ **Ordinal Encoding** আর কোনটিতে **One-Hot Encoding** দরকার — সেই সিদ্ধান্ত নিতে পারব।
- ভুল encoding-এর বিপদ বুঝব — Nominal data-তে Ordinal Encoding করলে model ভুল ক্রম শিখবে।


---

## 🧠 আমরা যা শিখেছি, সেই আলোকে কীভাবে চিন্তা করতে হবে?

প্রতিটি feature দেখে একটিমাত্র প্রশ্ন করতে হবে:

> **"এই values-গুলোর মধ্যে কি একটি স্বাভাবিক, অর্থপূর্ণ ক্রম (natural order) আছে?"**

| উত্তর | Type | Encoding |
|---|---|---|
| **হ্যাঁ** — একটি মান আরেকটির চেয়ে স্পষ্টতই "বেশি" বা "কম" | **Ordinal** | Ordinal Encoding |
| **না** — values শুধু আলাদা আলাদা category, কোনো ক্রম নেই | **Nominal** | One-Hot Encoding |

### তিনটি feature বিশ্লেষণ:

**(a) `T_shirt_size` = {S, M, L, XL}**
→ S < M < L < XL — স্পষ্ট ক্রম আছে (S সবচেয়ে ছোট, XL সবচেয়ে বড়) → **Ordinal**

**(b) `City` = {Dhaka, Chattogram, Rajshahi}**
→ কোনো city কি অন্যটির চেয়ে "বেশি"? না — এগুলো শুধু আলাদা location → **Nominal**

**(c) `Satisfaction` = {Low, Medium, High}**
→ Low < Medium < High — স্পষ্ট ক্রম আছে → **Ordinal**


---

## 🛠️ Problem Solve করার Approach

**Step 1:** তিনটি feature-এর data তৈরি করা।

**Step 2:** প্রতিটি feature Nominal না Ordinal সেটা justify করা।

**Step 3:** সঠিক encoding apply করে দেখানো — Ordinal feature-এ Ordinal Encoding, Nominal feature-এ One-Hot Encoding।

**Step 4:** ভুল encoding-এর বিপদ দেখানো — Nominal-এ Ordinal Encoding করলে কী হয়।


## Step 1: Data তৈরি করা

In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    'T_shirt_size': ['S', 'M', 'L', 'XL', 'M', 'S'],
    'City':         ['Dhaka', 'Chattogram', 'Rajshahi', 'Dhaka', 'Rajshahi', 'Chattogram'],
    'Satisfaction': ['Low', 'High', 'Medium', 'High', 'Low', 'Medium']
})

print(df.to_string(index=False))


T_shirt_size       City Satisfaction
           S      Dhaka          Low
           M Chattogram         High
           L   Rajshahi       Medium
          XL      Dhaka         High
           M   Rajshahi          Low
           S Chattogram       Medium


`pd.DataFrame` দিয়ে তিনটি feature-এর sample data তৈরি করা হয়েছে।
প্রতিটি column-এ সব unique value অন্তত একবার আছে।


---

## Step 2: Nominal না Ordinal — সিদ্ধান্ত ও Justification


In [2]:
classification = {
    'Feature':       ['T_shirt_size', 'City', 'Satisfaction'],
    'Type':          ['Ordinal', 'Nominal', 'Ordinal'],
    'Natural Order': ['S < M < L < XL', 'No order exists', 'Low < Medium < High'],
    'Encoding':      ['Ordinal Encoding', 'One-Hot Encoding', 'Ordinal Encoding']
}

result = pd.DataFrame(classification)
print(result.to_string(index=False))


     Feature    Type       Natural Order         Encoding
T_shirt_size Ordinal      S < M < L < XL Ordinal Encoding
        City Nominal     No order exists One-Hot Encoding
Satisfaction Ordinal Low < Medium < High Ordinal Encoding


**মূল পার্থক্য:**
- `T_shirt_size` ও `Satisfaction` → ক্রম আছে → **Ordinal** → Ordinal Encoding।
- `City` → ক্রম নেই → **Nominal** → One-Hot Encoding।


---

## Step 3a: Ordinal Encoding — `T_shirt_size` ও `Satisfaction`


In [3]:
# T_shirt_size: S=0, M=1, L=2, XL=3
tshirt_order = [['S', 'M', 'L', 'XL']]
enc_tshirt   = OrdinalEncoder(categories=tshirt_order)
df['Tshirt_Encoded'] = enc_tshirt.fit_transform(df[['T_shirt_size']]).astype(int)

# Satisfaction: Low=0, Medium=1, High=2
satisfaction_order = [['Low', 'Medium', 'High']]
enc_sat            = OrdinalEncoder(categories=satisfaction_order)
df['Satisfaction_Encoded'] = enc_sat.fit_transform(df[['Satisfaction']]).astype(int)

print(df[['T_shirt_size', 'Tshirt_Encoded', 'Satisfaction', 'Satisfaction_Encoded']].to_string(index=False))


T_shirt_size  Tshirt_Encoded Satisfaction  Satisfaction_Encoded
           S               0          Low                     0
           M               1         High                     2
           L               2       Medium                     1
          XL               3         High                     2
           M               1          Low                     0
           S               0       Medium                     1


`OrdinalEncoder(categories=...)` — `categories` parameter-এ list-of-lists দিতে হয়।
ভেতরের list-এ **ক্রম অনুযায়ী** category সাজানো থাকে — সবচেয়ে ছোট প্রথমে।
`.astype(int)` → output default float, তাই integer-এ convert করা হয়েছে।


---

## Step 3b: One-Hot Encoding — `City`


In [4]:
city_dummies = pd.get_dummies(df['City'], prefix='City')

df_final = pd.concat([df[['City']], city_dummies], axis=1)
print(df_final.to_string(index=False))


      City  City_Chattogram  City_Dhaka  City_Rajshahi
     Dhaka            False        True          False
Chattogram             True       False          False
  Rajshahi            False       False           True
     Dhaka            False        True          False
  Rajshahi            False       False           True
Chattogram             True       False          False


`pd.get_dummies()` → প্রতিটি unique city-র জন্য একটি binary column তৈরি করে।
`prefix='City'` → নতুন column-এর নাম হয় `City_Dhaka`, `City_Chattogram`, `City_Rajshahi`।
প্রতিটি row-এ ঠিক একটি column-এ **1**, বাকি সব **0**।


---

## Step 4: ভুল Encoding-এর বিপদ — Nominal-এ Ordinal Encoding করলে কী হয়?


In [5]:
# WRONG: applying ordinal encoding to City (a nominal feature)
wrong_enc = OrdinalEncoder()
df['City_WRONG_Encoded'] = wrong_enc.fit_transform(df[['City']]).astype(int)

print("WRONG encoding for City (Nominal feature):")
print(df[['City', 'City_WRONG_Encoded']].to_string(index=False))
print()
print("Implied wrong order:")
mapping = dict(zip(df['City'], df['City_WRONG_Encoded']))
for city, code in sorted(mapping.items(), key=lambda x: x[1]):
    print(f"  {code} = {city}")
print()
print("Problem: Model will think Chattogram(0) < Dhaka(1) < Rajshahi(2)")
print("         But cities have NO meaningful numeric relationship!")


WRONG encoding for City (Nominal feature):
      City  City_WRONG_Encoded
     Dhaka                   1
Chattogram                   0
  Rajshahi                   2
     Dhaka                   1
  Rajshahi                   2
Chattogram                   0

Implied wrong order:
  0 = Chattogram
  1 = Dhaka
  2 = Rajshahi

Problem: Model will think Chattogram(0) < Dhaka(1) < Rajshahi(2)
         But cities have NO meaningful numeric relationship!


**কেন এটা বিপজ্জনক?**

`OrdinalEncoder` alphabetical order-এ encode করে:
- Chattogram = 0, Dhaka = 1, Rajshahi = 2

এখন model ভাববে: **Rajshahi (2) = Dhaka (1) × 2** — অর্থাৎ Rajshahi Dhaka-র "দ্বিগুণ"!

এই ভুল numeric relationship model-এর prediction-কে **সম্পূর্ণ বিকৃত** করে দেবে।

> ✅ **নিয়ম:** Nominal feature-এ সবসময় **One-Hot Encoding** — কখনোই Ordinal Encoding নয়।


## Final: Complete Encoded Dataset

In [6]:
final = pd.concat([
    df[['T_shirt_size', 'Tshirt_Encoded',
        'Satisfaction', 'Satisfaction_Encoded']],
    city_dummies
], axis=1)

print("Complete encoded dataset (correct encodings only):")
print(final.to_string(index=False))


Complete encoded dataset (correct encodings only):
T_shirt_size  Tshirt_Encoded Satisfaction  Satisfaction_Encoded  City_Chattogram  City_Dhaka  City_Rajshahi
           S               0          Low                     0            False        True          False
           M               1         High                     2             True       False          False
           L               2       Medium                     1            False       False           True
          XL               3         High                     2            False        True          False
           M               1          Low                     0            False       False           True
           S               0       Medium                     1             True       False          False


Final dataset-এ কোনো text নেই — সম্পূর্ণ numeric।
- `T_shirt_size` → একটি column (0–3)।
- `Satisfaction` → একটি column (0–2)।
- `City` → তিনটি binary column।

এই dataset এখন সরাসরি ML model-এ দেওয়ার জন্য প্রস্তুত।


---